运行约定：以本 Notebook 所在目录作为工作目录，从第一单元顺序运行。首个代码单元仅建立数据与输出目录。历史输出保留，不代表当前环境已验证。


In [ ]:
from pathlib import Path
Path('../../data').mkdir(parents=True, exist_ok=True)
Path('../../outputs/paddle').mkdir(parents=True, exist_ok=True)


In [1]:
import warnings
warnings.filterwarnings("ignore")

# 一、自然语言情感分析任务为例

人类自然语言具有高度的复杂性，相同的对话在不同的情景，不同的情感，不同的人演绎，表达的效果往往也会迥然不同。

人类自然语言不只具有复杂性，同时也蕴含着丰富的情感色彩：表达人的情绪（如悲伤、快乐）、表达人的心情（如倦怠、忧郁）、表达人的喜好（如喜欢、讨厌）、表达人的个性特征和表达人的立场等等。

简单的说，我们可以将情感分析（sentiment classification）任务定义为一个分类问题，即指定一个文本输入，机器通过对文本进行分析、处理、归纳和推理后自动输出结论，如**图**所示。
<center><img src="../assets/03-lstm-sentiment-1.jpg" width="800" ></center>
<br><center>图：情感分析任务</center></br>

通常情况下，人们把情感分析任务看成一个三分类问题，如 **图** 所示：
<br></br>
<center><img src="../assets/03-lstm-sentiment-2.png" width="600" ></center>
<br><center>图：情感分析任务</center></br>

- **正向：** 表示正面积极的情感，如高兴，幸福，惊喜，期待等。
- **负向：** 表示负面消极的情感，如难过，伤心，愤怒，惊恐等。
- **其他：** 其他类型的情感。

在情感分析任务中，研究人员除了分析句子的情感类型外，还细化到以句子中具体的“方面”为分析主体进行情感分析（aspect-level），如下：

> 这个薯片口味有点咸，太辣了，不过口感很脆。

关于薯片的口味方面是一个负向评价（咸，太辣），然而对于口感方面却是一个正向评价（很脆）。

> 我很喜欢夏威夷，就是这边的海鲜太贵了。

关于夏威夷是一个正向评价（喜欢），然而对于夏威夷的海鲜却是一个负向评价（价格太贵）。


## 1.1 情感分析任务的挑战

我们已经学习了通过把每个单词转换成向量的方式，可以完成单词语义计算任务。  
然而由于自然语言变幻莫测，我们在使用神经网络处理句子的时候，往往会遇到如下两类问题：  

- **变长句子的规整化处理，：** 自然语言句子往往是变长的，不同的句子长度可能差别很大。主流的神经网络（如RNN, LSTM, Transformer）在计算时，其输入通常要求是一个规整的多维张量，长度是固定的。对于一个batch的句子来说，最自然的表示就是[batch_size, sequence_length, feature_dim]的三维张量。这就要求sequence_length是固定的。

- **词序敏感的语义建模：** 自然语言句子往往对结构非常敏感，有时稍微颠倒单词的顺序都可能改变这句话的意思，比如：

  >你等一下我做完作业就走。
  >
  >我等一下你做完工作就走。

  >我不爱吃你做的饭。
  >
  >你不爱吃我做的饭。
  
  >我瞅你咋地。
  >
  >你瞅我咋地。
  
因此，我们需要找到一个可以考虑词和词之间顺序（关系）的神经网络，用于更好地实现自然语言句子建模。

## 1.2 变长句子的规整化处理

在使用神经网络处理变长数据时，需要先设置一个全局变量max_seq_len，再对语料中的句子进行处理，将不同的句子组成mini-batch，用于神经网络学习和处理。

**1. 设置全局变量**

设定一个全局变量max_seq_len，用来控制神经网络最大可以处理文本的长度。我们可以先观察语料中句子的分布，再设置合理的max_seq_len值，以最高的性价比完成句子分类任务。

**2. 对语料中的句子进行处理**

我们通常采用 **截断+填充** 的方式，对语料中的句子进行处理，将不同长度的句子组成mini-batch，以便让句子转换成一个张量给神经网络进行计算，如 **图** 所示。
<br></br>
<center><img src="../assets/03-lstm-sentiment-3.png" width="400" ></center>
<br><center>图：变长数据处理</center></br>

* 对于长度超过max_seq_len的句子，通常会把这个句子进行截断，以便可以输入到一个张量中。句子截断是有技巧的，有时截取句子的前一部分会比后一部分好，有时则恰好相反。
  - **前向截断：** “晚饭， 真， 难， 以， 下， 咽”
  - **后向截断：**“今天， 的， 晚饭， 真， 难， 以”
* 对于句子长度不足max_seq_len的句子，我们一般会使用一个特殊的词语对这个句子进行填充，这个过程称为Padding。假设给定一个句子“我，爱，人工，智能”，max_seq_len=6，那么可能得到两种填充方式：
  - **前向填充：** “[pad]，[pad]，我，爱，人工，智能”
  - **后向填充：**“我，爱，人工，智能，[pad]，[pad]”

同样，不同的填充方式也对网络训练效果有一定影响。一般来说，后向填充是更常用的选择。

## 1.3 词序敏感的语义建模

向量化中，学习了如何学习每个单词的语义信息，从上面的举例中我们也会观察到，一个句子中词的顺序往往对这个句子的整体语义有重要的影响。因此，在刻画整个句子的语义信息过程中，不能撇开顺序信息。如果简单粗暴地把这个句子中所有词的向量做加和，会使得我们的模型无法区分句子的真实含义，例如：

>我不爱吃你做的饭。
>
>你不爱吃我做的饭。

一个有趣的想法，把一个自然语言句子看成一个序列，把自然语言的生成过程看成是一个序列生成的过程。例如对于句子“我，爱，人工，智能”，这句话的生成概率$\text{P}(\text{我，爱，人工，智能})$可以被表示为：

$\text{P}(\text{我，爱，人工，智能})=\text{P}(我|\text{<s>})*\text{P}(爱|\text{<s>，我})* \text{P}(人工|\text{<s>，我，爱})* \text{P}(智能|\text{<s>，我，爱，人工})* \text{P}(\text{</s>}|\text{<s>，我，爱，人工，智能})$

其中$\text{<s>}$和$\text{</s>}$是两个特殊的不可见符号，表示一个句子在逻辑上的开始和结束。

上面的公式把一个句子的生成过程建模成一个序列的决策过程，这就是香农在1950年左右提出的使用马尔可夫过程建模自然语言的思想。使用序列的视角看待和建模自然语言有一个明显的好处，那就是在对每个词建模的过程中，都有机会去学习这个词和之前生成的词之间的关系，并利用这种关系更好地处理自然语言。如 **图** 所示，生成句子“我，爱，人工”后，“智能”在下一步生成的概率就变得很高了，因为“人工智能”经常同时出现。
<center><img src="../assets/03-lstm-sentiment-4.png" width="600" ></center>
<br><center>图：自然语言生成过程示意图</center></br>

通过考虑句子内部的序列关系，我们就可以清晰地区分“我不爱吃你做的菜”和“你不爱吃我做的菜”这两句话之间的联系与不同了。

事实上，目前大多数成功的自然语言模型都建立在对句子的序列化建模上。下面让我们学习两个经典的序列化建模模型：循环神经网络（Recurrent Neural Network，RNN）和长短时记忆网络（Long Short-Term Memory，LSTM）。

**循环神经网络RNN和长短时记忆网络LSTM的设计思考**

卷积神经网络的设计具备适合视觉任务“局部视野”特点，是因为视觉信息是局部有效的。例如在一张图片的1/4区域上有一只小猫，如果将图片3/4的内容遮挡，人类依然可以判断这是一只猫。

与此类似，RNN和LSTM的设计初衷是部分场景神经网络需要有“记忆”能力才能解决的任务。在自然语言处理任务中，往往一段文字中某个词的语义可能与前一段句子的语义相关，只有记住了上下文的神经网络才能很好的处理句子的语义关系。例如：
> 我一边吃着苹果，一边玩着苹果手机。

网络只有正确的记忆两个“苹果”的上下文“吃着”和“玩着…手机”，才能正确的识别两个苹果的语义，分别是水果和手机品牌。如果网络没有记忆功能，那么两个“苹果”只能归结到更高概率出现的语义上，得到一个相同的语义输出，这显然是不合理的。

如何设计神经网络的记忆功能呢？我们先了解下RNN网络是如何实现具备记忆功能的。RNN相当于将神经网络单元进行了横向连接，处理前一部分输入的RNN单元不仅有正常的模型输出，还会输出“记忆”传递到下一个RNN单元。而处于后一部分的RNN单元，不仅仅有来自于任务数据的输入，同时会接收从前一个RNN单元传递过来的记忆输入，这样就使得整个神经网络具备了“记忆”能力。

但是RNN网络只是初步实现了“记忆”功能，在此基础上科学家们又发明了一些RNN的变体，来加强网络的记忆能力。但RNN对“记忆”能力的设计是比较粗糙的，当网络处理的序列数据过长时，累积的内部信息就会越来越复杂，直到超过网络的承载能力，通俗的说“事无巨细的记录，总有一天大脑会崩溃”。为了解决这个问题，科学家巧妙的设计了一种记忆单元，称之为“长短时记忆网络（Long Short-Term Memory，LSTM）”。在每个处理单元内部，加入了输入门、输出门和遗忘门的设计，三者有明确的任务分工：

* 输入门：控制有多少输入信号会被融合；
* 遗忘门：控制有多少过去的记忆会被遗忘；
* 输出门：控制多少处理后的信息会被输出；

三者的作用与人类的记忆方式有异曲同工之处，即：

* 与当前任务无关的信息会直接过滤掉，如非常专注的开车时，人们几乎不注意沿途的风景；
* 过去记录的事情不一定都要永远记住，如令人伤心或者不重要的事，通常会很快被淡忘；
* 根据记忆和现实观察进行决策，如开车时会结合记忆中的路线和当前看到的路标，决策转弯或不做任何动作。

了解了这些关于网络设计的本质，下面进入实现方案的细节。

# 二、使用LSTM完成情感分析任务 

借助长短时记忆网络，我们可以非常轻松地完成情感分析任务。

对于每个句子，我们首先通过截断和填充的方式，把这些句子变成固定长度的向量。然后，利用长短时记忆网络，从左到右开始阅读每个句子。在完成阅读之后，我们使用长短时记忆网络的最后一个输出记忆，作为整个句子的语义信息，并直接把这个向量作为输入，送入一个分类层进行分类，从而完成对情感分析问题的神经网络建模。
<center><img src="../assets/03-lstm-sentiment-5.png" width="400" ></center>
<br><center>图：LSTM完成情感分析任务流程</center></br>




接下来让我们看看如何使用飞桨实现一个基于长短时记忆网络的情感分析模型。不同深度学习模型的训练过程基本一致，流程如下：

1. **数据处理**：选择需要使用的数据，并做好必要的预处理工作。

2. **网络定义**：使用飞桨定义好网络结构，包括输入层，中间层，输出层，损失函数和优化算法。

3. **模型训练**：将准备好的训练集数据送入神经网络进行学习，并观察学习的过程是否正常，如损失函数值是否在降低，也可以打印一些中间步骤的结果出来等。

4. **模型评估**：使用测试集数据测试训练好的神经网络，看看训练效果如何。
4. **模型预测**：取出一条文本数据，放入模型后进行预测，观察预测的情感倾向值。

在数据处理前，需要先加载飞桨平台（如果用户在本地使用，请确保已经安装飞桨）。

In [11]:
import os
import random
import numpy as np
import pandas as pd
# 导入Paddle的API
import paddle
import paddle.nn as nn
import paddle.nn.functional as F
from paddle.nn import LSTM, Embedding, Dropout, Linear

from paddlenlp.datasets import load_dataset
from paddlenlp.utils.downloader import get_path_from_url

# 指定用第 0 号 GPU，如果想用第 1 张卡就写 1
# paddle.set_device('gpu:0')

import warnings
warnings.filterwarnings("ignore")

AttributeError: module 'paddle' has no attribute 'pir'

## 2.1 直接使用paddle内置预训练的模型

In [ ]:
import os
import shutil
URL = "https://bj.bcebos.com/paddlenlp/datasets/ChnSentiCorp.zip"
# 如果数据集不存在，就下载数据集并解压
if(not os.path.exists('ChnSentiCorp.zip')):
    get_path_from_url(URL,root_dir='../../data/')

# 修复嵌套目录：如果存在 ../../data/ChnSentiCorp/ChnSentiCorp，则上移
nested_path = '../../data/ChnSentiCorp/ChnSentiCorp'
if os.path.exists(nested_path):
    # 临时移动到别处，删除外层空壳，再移回来
    shutil.move(nested_path, '../../data/ChnSentiCorp_temp')
    shutil.rmtree('../../data/ChnSentiCorp')
    shutil.move('../../data/ChnSentiCorp_temp', '../../data/ChnSentiCorp')
    print("已修复目录嵌套")

In [78]:
def read(split='train'):
    data_dict={'train':'../../data/ChnSentiCorp/train.tsv',
                "dev":'../../data/ChnSentiCorp/dev.tsv',
                'test':'../../data/ChnSentiCorp/test.tsv'}
    with open(data_dict[split],'r') as f:
        head = None
        # 一行一行的读取数据
        for line in f.readlines():
            data = line.strip().split("\t")
            # 跳过第一行，因为第一行是列名
            if not head:
                head = data
            else:
                # 从第二行还是一行一行的返回数据
                if split == 'train':
                    label, text = data
                    yield {"text": text, "label": label, "qid": ''}
                elif split == 'dev':
                    qid, label, text = data
                    yield {"text": text, "label": label, "qid": qid}
                elif split == 'test':
                    qid, text = data
                    yield {"text": text, "label": '', "qid": qid}


train_ds= load_dataset(read, split="train",lazy=False)
dev_ds= load_dataset(read, split="dev",lazy=False)
test_ds= load_dataset(read, split="test",lazy=False)

In [79]:
for data in dev_ds.data[:5]:
    print(data)

{'text': '這間酒店環境和服務態度亦算不錯,但房間空間太小~~不宣容納太大件行李~~且房間格調還可以~~ 中餐廳的廣東點心不太好吃~~要改善之~~~~但算價錢平宜~~可接受~~ 西餐廳格調都很好~~但吃的味道一般且令人等得太耐了~~要改善之~~', 'label': '1', 'qid': '0'}
{'text': '<荐书> 推荐所有喜欢<红楼>的红迷们一定要收藏这本书,要知道当年我听说这本书的时候花很长时间去图书馆找和借都没能如愿,所以这次一看到当当有,马上买了,红迷们也要记得备货哦!', 'label': '1', 'qid': '1'}
{'text': '商品的不足暂时还没发现，京东的订单处理速度实在.......周二就打包完成，周五才发货...', 'label': '0', 'qid': '2'}
{'text': '２００１年来福州就住在这里，这次感觉房间就了点，温泉水还是有的．总的来说很满意．早餐简单了些．', 'label': '1', 'qid': '3'}
{'text': '不错的上网本，外形很漂亮，操作系统应该是个很大的 卖点，电池还可以。整体上讲，作为一个上网本的定位，还是不错的。', 'label': '1', 'qid': '4'}


In [80]:
import random
import numpy as np
import paddle
import paddle.nn.functional as F
from paddlenlp.transformers import ErnieTokenizer, ErnieForSequenceClassification

# 1. 初始化tokenizer和模型（添加这部分代码）
model_name = "ernie-3.0-medium-zh"  # ERNIE 3.0-Medium-zh​ 是百度文心大模型家族中的轻量化中文预训练模型
tokenizer = ErnieTokenizer.from_pretrained(model_name)
model = ErnieForSequenceClassification.from_pretrained(model_name, num_classes=2)

# 2. 随机选择一个样本
random_index = random.randint(0, len(dev_ds) - 1)
sample = dev_ds[random_index]
text = sample['text']
true_label = sample['label']

# 3. 标签映射
label_map = {0: '负面评价', 1: '正面评价'}

# 4. 文本编码（使用正确的编码方式）
encoded_input = tokenizer.encode(text)

# 5. 转换为Tensor
input_ids = paddle.to_tensor([encoded_input['input_ids']])
token_type_ids = paddle.to_tensor([encoded_input['token_type_ids']])

# 6. 模型预测
model.eval()
with paddle.no_grad():
    logits = model(input_ids, token_type_ids)
    probs = F.softmax(logits, axis=1)

# 7. 获取预测结果
predicted_label_id = np.argmax(probs.numpy(), axis=-1)[0]
predicted_label = label_map[predicted_label_id]
confidence = probs.numpy()[0][predicted_label_id]

# 8. 真实标签
true_label_text = label_map[int(true_label)]

# 9. 对比结果
is_correct = predicted_label_id == int(true_label)
status = "✓ 正确" if is_correct else "✗ 错误"

# 10. 输出结果
print(f"样本索引: {random_index}")
print(f"文本内容: {text}")
print(f"真实标签: {true_label_text} ({true_label})")
print(f"预测标签: {predicted_label} ({predicted_label_id})")
print(f"置信度: {confidence:.4f}")
print(f"预测状态: {status}")

[2026-01-09 21:57:18,563] [    INFO] - Already cached /root/.paddlenlp/models/ernie-3.0-medium-zh/ernie_3.0_medium_zh_vocab.txt
[2026-01-09 21:57:18,587] [    INFO] - tokenizer config file saved in /root/.paddlenlp/models/ernie-3.0-medium-zh/tokenizer_config.json
[2026-01-09 21:57:18,588] [    INFO] - Special tokens file saved in /root/.paddlenlp/models/ernie-3.0-medium-zh/special_tokens_map.json
[2026-01-09 21:57:18,591] [    INFO] - Already cached /root/.paddlenlp/models/ernie-3.0-medium-zh/model_state.pdparams
[2026-01-09 21:57:18,591] [    INFO] - Loading weights file model_state.pdparams from cache at /root/.paddlenlp/models/ernie-3.0-medium-zh/model_state.pdparams
[2026-01-09 21:57:18,850] [    INFO] - Loaded weights file from disk, setting weights to model.
[2026-01-09 21:57:19,183] [ WARNING] - Some weights of the model checkpoint at ernie-3.0-medium-zh were not used when initializing ErnieForSequenceClassification: ['ernie.encoder.layers.6.self_attn.out_proj.bias', 'ernie.enco

样本索引: 861
文本内容: 换XP搞费劲的 颜色我一直以为是黑色的 害的我弄了一层膜上去 声音的触摸键不灵 散热不是很好 风扇声音太大了
真实标签: 负面评价 (0)
预测标签: 负面评价 (0)
置信度: 0.5657
预测状态: ✓ 正确


## 2.2 手动训练

### 2.2.1 数据处理

    
<center><img src="../assets/03-lstm-sentiment-6.png" width="800px" ></center>
<br><center>数据处理流程</center></br>
    
上图显示的是数据处理的流程，首先是输入文本，根据输入的文本数据进行分词，分词完以后构造词表，然后根据词表把分词后的文本映射成id的形式，然后再组装成batch形式的数据。

1.**分词**：使用结巴分词工具，把句子切分成单词的形式。

2.**构造词表**：将单词形式的句子，统计频率，按照频率取topK个单词，作为词表，剩下的低频单词舍弃，并给每个单词编一个号。

3.**转换成id**：主要是利用构造的词汇表，对分好词的数据映射成相应的id的形式。

4.**组装batch**：组装成一个batch数据（多条数据堆叠在一起），但是每个句子有长有短，对于较短的句子，我们需要补0对齐。

#### 加载数据集

首先，需要下载语料用于模型训练和评估效果。我们使用的是ChnSentiCorp的酒店评论数据，这个数
据集是一个开源的中文数据集，由训练数据，验证集和测试数据组成。训练集的数量是9601条，验证集的数量是1201条，测试集的数量是1201条。每个数据表示的是用户对酒店的真实评价，以及用户对酒店的情感倾向（是正向还是负向），数据集下载的代码如下：

In [81]:
import os
import random
import numpy as np
import pandas as pd
# 导入Paddle的API
import paddle
import paddle.nn as nn
import paddle.nn.functional as F
from paddle.nn import LSTM, Embedding, Dropout, Linear

from paddlenlp.datasets import load_dataset
from paddlenlp.utils.downloader import get_path_from_url

# 指定用第 0 号 GPU，如果想用第 1 张卡就写 1
paddle.set_device('gpu:0')

import warnings
warnings.filterwarnings("ignore")

In [82]:
# URL = "https://bj.bcebos.com/paddlenlp/datasets/ChnSentiCorp.zip"
# # 如果数据集不存在，就下载数据集并解压
# if(not os.path.exists('ChnSentiCorp.zip')):
#     get_path_from_url(URL,root_dir='../../data/')

[2026-01-09 21:57:19,269] [    INFO] - Found DataSet/ChnSentiCorp.zip
[2026-01-09 21:57:19,271] [    INFO] - Decompressing DataSet/ChnSentiCorp.zip...


In [83]:
# !unzip ../../data/ChnSentiCorp.zip -d ../../data

In [84]:
def read(split='train'):
    data_dict={'train':'../../data/ChnSentiCorp/train.tsv',
                "dev":'../../data/ChnSentiCorp/dev.tsv',
                'test':'../../data/ChnSentiCorp/test.tsv'}
    with open(data_dict[split],'r') as f:
        head = None
        # 一行一行的读取数据
        for line in f.readlines():
            data = line.strip().split("\t")
            # 跳过第一行，因为第一行是列名
            if not head:
                head = data
            else:
                # 从第二行还是一行一行的返回数据
                if split == 'train':
                    label, text = data
                    yield {"text": text, "label": label, "qid": ''}
                elif split == 'dev':
                    qid, label, text = data
                    yield {"text": text, "label": label, "qid": qid}
                elif split == 'test':
                    qid, text = data
                    yield {"text": text, "label": '', "qid": qid}


train_ds= load_dataset(read, split="train",lazy=False)
dev_ds= load_dataset(read, split="dev",lazy=False)
test_ds= load_dataset(read, split="test",lazy=False)

接下来，将数据集加载到程序中，并打印一小部分数据观察一下数据集的特点，代码如下：

In [85]:
for data in train_ds.data[:5]:
    print(data)

{'text': '选择珠江花园的原因就是方便，有电动扶梯直接到达海边，周围餐馆、食廊、商场、超市、摊位一应俱全。酒店装修一般，但还算整洁。 泳池在大堂的屋顶，因此很小，不过女儿倒是喜欢。 包的早餐是西式的，还算丰富。 服务吗，一般', 'label': '1', 'qid': ''}
{'text': '15.4寸笔记本的键盘确实爽，基本跟台式机差不多了，蛮喜欢数字小键盘，输数字特方便，样子也很美观，做工也相当不错', 'label': '1', 'qid': ''}
{'text': '房间太小。其他的都一般。。。。。。。。。', 'label': '0', 'qid': ''}
{'text': '1.接电源没有几分钟,电源适配器热的不行. 2.摄像头用不起来. 3.机盖的钢琴漆，手不能摸，一摸一个印. 4.硬盘分区不好办.', 'label': '0', 'qid': ''}
{'text': '今天才知道这书还有第6卷,真有点郁闷:为什么同一套书有两种版本呢?当当网是不是该跟出版社商量商量,单独出个第6卷,让我们的孩子不会有所遗憾。', 'label': '1', 'qid': ''}


text表示的是评论文本，label表示的是标签，1表示文本的情感倾向是正向的，0表示文本的情感倾向是负向的。qid表示数据的编号，qid出现在测试集中，在训练集合里面数据的编号没有，所以qid为空。

#### 构造词表

读取数据后，需要构造一个词典，把每个词都转化成一个ID，以便于神经网络训练。代码如下：

------
**注意：**

在代码中我们使用了一个特殊的单词"[UNK]"，用于表示词表中没有覆盖到的词。之所以使用"[UNK]"这个符号，是为了处理某一些词，在测试数据中有，但训练数据没有的现象。

------

In [86]:
from collections import defaultdict
import jieba
from paddlenlp.data import Vocab

def build_vocab(texts,
                stopwords=[],
                num_words=None,
                min_freq=10,
                unk_token="[UNK]",
                pad_token="[PAD]"):
    word_counts = defaultdict(int)
    for text in texts:
        if not text:
            continue
        # # 统计词频
        # 一般来说，在自然语言处理中，需要先对语料进行切词，英文可以使用空格把每个句子切成若干词的序列
        # 对于中文则需要使用结巴分词进行切分
        for word in jieba.cut(text):
            if word in stopwords:
                continue
            word_counts[word] += 1
    # 过滤掉词频小于min_freq的单词
    wcounts = []
    for word, count in word_counts.items():
        if count < min_freq:
            continue
        wcounts.append((word, count))
    # 把单词按照词频从大到小进行排序
    wcounts.sort(key=lambda x: x[1], reverse=True)
    # 把对齐的字符和unk字符加入到词表中
    if num_words is not None and len(wcounts) > (num_words - 2):
        wcounts = wcounts[:(num_words - 2)]
    # pad字符和unk字符
    sorted_voc = [pad_token, unk_token]
    sorted_voc.extend(wc[0] for wc in wcounts)
    # 给每个单词一个编号
    word_index = dict(zip(sorted_voc, list(range(len(sorted_voc)))))
    return word_index

texts = []
for data in train_ds:
    texts.append(data["text"])
for data in dev_ds:
    texts.append(data["text"])

# 以下停用词仅用作示例，具体停用词的选择需要根据具体语料库调整。
stopwords = set(["的", "吗", "吧", "呀", "呜", "呢", "呗"])
# 构建词汇表
word2idx = build_vocab(
    texts, stopwords, min_freq=5, unk_token="[UNK]", pad_token="[PAD]")
vocab = Vocab.from_dict(word2idx, unk_token="[UNK]", pad_token="[PAD]")
# 保存词汇表
res=vocab.to_json("../../outputs/paddle/vocab.json")

#### 分词并转换成id

在完成word2id词典假设之后，我们还需要进一步处理原始语料，把语料中的所有句子都处理成ID序列，代码如下：

在转换成id之前，我们先通过下面的示例认识一下JiebaTokenizer：

In [87]:
from paddlenlp.data import JiebaTokenizer

def get_idx_from_word(word, word_to_idx, unk_word):
    if word in word_to_idx:
        return word_to_idx[word]
    return word_to_idx[unk_word]

# 假设vocab是已加载的词汇表
tokenizer = JiebaTokenizer(vocab)
text = '选择珠江花园的原因就是方便，有电动扶梯直接到达海边，周围餐馆、食廊、商场、超市、摊位一应俱全。'

# 方法1: 手动分词并转换
segment_text = tokenizer.cut(text)
input_ids_manual = [get_idx_from_word(word, vocab.token_to_idx, vocab.unk_token) for word in segment_text]

# 方法2: 直接使用encode方法
input_ids_encode = tokenizer.encode(text)

print("=" * 60)
print("原始文本: {}".format(text))
print("=" * 60)
print("分词结果: {}".format(segment_text))
print("手动转换ID: {}".format(input_ids_manual))
print("编码结果ID: {}".format(input_ids_encode))
print("=" * 60)

原始文本: 选择珠江花园的原因就是方便，有电动扶梯直接到达海边，周围餐馆、食廊、商场、超市、摊位一应俱全。
分词结果: ['选择', '珠江', '花园', '的', '原因', '就是', '方便', '，', '有', '电动', '扶梯', '直接', '到达', '海边', '，', '周围', '餐馆', '、', '食廊', '、', '商', '场', '、', '超市', '、', '摊位', '一应', '俱全', '。']
手动转换ID: [198, 1, 1595, 1, 406, 38, 105, 2, 16, 1, 1, 349, 987, 1699, 2, 663, 3103, 26, 1, 26, 1, 6212, 26, 1453, 26, 1, 1, 1, 3]
编码结果ID: [198, 1, 1595, 1, 406, 38, 105, 2, 16, 1, 1, 349, 987, 1699, 2, 663, 3103, 26, 1, 26, 1, 6212, 26, 1453, 26, 1, 1, 1, 3]


从上述的输出可以看到JiebaTokenizer的encode的作用就是对文本进行分词，然后根据词汇表把文本转换成id的形式。

In [88]:
from functools import partial

# 把语料转换为id序列
def convert_example(example, tokenizer, is_test=False):
    # 将句子中的词逐个替换成id，如果句子中的词不在词表内，则替换成[UNK]
    input_ids = tokenizer.encode(example["text"])
    # 计算出数据转换成id后的长度，并转换成numpy的格式
    valid_length = np.array(len(input_ids), dtype='int64')
    # 把id形式的数据转换成numpy的形式
    input_ids = np.array(input_ids, dtype='int64')
    # 训练集需要label
    if not is_test:
        label = np.array(example["label"], dtype="int64")
        return input_ids, valid_length, label
    else:
        # 测试集不需要label
        return input_ids, valid_length

# partial函数的意思是把tokenizer=tokenizer, is_test=False赋值给当前的convert_example函数
trans_fn = partial(convert_example, tokenizer=tokenizer, is_test=False)
# 训练数据转换成id的形式
train_ds = train_ds.map(trans_fn)
# 验证集转换成id的形式
dev_ds = dev_ds.map(trans_fn)


#### 组装成mini-batch

接下来，我们就可以开始把原始语料中的每个句子通过截断和填充，转换成一个固定长度的句子，并将所有数据整理成mini-batch，用于训练模型，代码如下：

在组装minit-batch数据之前，我们先介绍一下需要使用的API，Stack, Pad, Tuple，Stack的功能是把向量堆叠在一起，Pad的功能是把不同长度的向量填充补齐，Tuple的功能是将多个batchify函数（比如Pad，Stack）包装在一起，示例如下：

In [89]:
from paddlenlp.data import Stack, Pad, Tuple

# 构建a，b，c三个向量
a = [1, 2, 3, 4]
b = [3, 4, 5, 6]
c = [5, 6, 7, 8]
result = Stack()([a, b, c])
print("堆叠（Stacked）数据后 : \n", result)
print()

# 构建a，b，c三个向量
a = [1, 2, 3, 4]
b = [5, 6, 7]
c = [8, 9]
result = Pad(pad_val=0)([a, b, c])
print("对齐（Padded）数据后: \n", result)
print()

# 构造一个小的样本，包含输入id和label
data = [
        [[1, 2, 3, 4], [1]],
        [[5, 6, 7], [0]],
        [[8, 9], [1]],
       ]
batchify_fn = Tuple(Pad(pad_val=0), Stack())
ids, labels = batchify_fn(data)
print("id的输出: \n", ids)
print()
print("标签的输出: \n", labels)
print()

堆叠（Stacked）数据后 : 
 [[1 2 3 4]
 [3 4 5 6]
 [5 6 7 8]]

对齐（Padded）数据后: 
 [[1 2 3 4]
 [5 6 7 0]
 [8 9 0 0]]

id的输出: 
 [[1 2 3 4]
 [5 6 7 0]
 [8 9 0 0]]

标签的输出: 
 [[1]
 [0]
 [1]]



In [90]:
batch_size = 64
batchify_fn = lambda samples, fn=Tuple(
    Pad(axis=0, pad_val=vocab.token_to_idx.get('[PAD]', 0)),  # 表示在一个mini-batch与最长的那条数据对齐，长度不够的话用0来补齐
    Stack(dtype="int64"),  # seq len
    Stack(dtype="int64")  # label
): [data for data in fn(samples)]

# 训练集的sampler，迭代式获取mini-batch的样本下标数组，数组长度与 batch_size 一致
train_sampler = paddle.io.BatchSampler(
            dataset=train_ds, batch_size=batch_size, shuffle=True)
# 测试集的sampler，迭代式获取mini-batch的样本下标数组，数组长度与 batch_size 一致
test_sampler = paddle.io.BatchSampler(
            dataset=dev_ds, batch_size=batch_size, shuffle=True)
# 使用paddle.io.DataLoader接口多线程异步加载数据
# DataLoader根据 batch_sampler 给定的顺序迭代一次给定的 dataset
train_loader = paddle.io.DataLoader(
        train_ds, batch_sampler=train_sampler, collate_fn=batchify_fn)
# 使用验证集作为测试集，因为验证集包含label。而原来的测试集没有label，不方便算指标
test_loader = paddle.io.DataLoader(
        dev_ds, batch_sampler=test_sampler, collate_fn=batchify_fn)
# 打印输出一个mini-batch的数据
for idx,item in enumerate(train_loader):
    if(idx==0):
        print(item)
        # break

[Tensor(shape=[64, 270], dtype=int64, place=Place(gpu_pinned), stop_gradient=True,
       [[21  , 94  , 2   , ..., 0   , 0   , 0   ],
        [784 , 75  , 1375, ..., 0   , 0   , 0   ],
        [13  , 7774, 381 , ..., 0   , 0   , 0   ],
        ...,
        [432 , 120 , 3863, ..., 0   , 0   , 0   ],
        [51  , 216 , 20  , ..., 0   , 0   , 0   ],
        [21  , 409 , 172 , ..., 23  , 8475, 59  ]]), Tensor(shape=[64], dtype=int64, place=Place(gpu_pinned), stop_gradient=True,
       [36 , 54 , 163, 13 , 116, 42 , 28 , 36 , 30 , 122, 41 , 32 , 26 , 34 ,
        39 , 91 , 106, 126, 71 , 47 , 65 , 55 , 101, 25 , 47 , 205, 92 , 49 ,
        67 , 39 , 31 , 243, 35 , 203, 37 , 36 , 52 , 120, 38 , 36 , 37 , 32 ,
        131, 53 , 21 , 35 , 130, 67 , 51 , 125, 96 , 129, 136, 98 , 81 , 31 ,
        21 , 140, 125, 129, 33 , 116, 26 , 270]), Tensor(shape=[64], dtype=int64, place=Place(gpu_pinned), stop_gradient=True,
       [0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0,


上面是迭代器的输出，可以看到输出的数据第一个Tensor中的batch size的大小是64，每条数据的长度是432；  
第二个Tensor存放的是每条数据实际长度；  
第三个Tensor表示的是每条数据的label，总共64个label。  

### 2.2.2 网络定义

在讲解卷积神经网络的的章节，我们详细列出了每一种神经网络使用基础算子拼装的详细网络配置，但实际上对于一些常用的网络结构，飞桨框架提供了现成的中高层函数支持。下面用于情感分析的长短时记忆模型就使用 [paddle.nn.LSTM](https://www.paddlepaddle.org.cn/documentation/docs/zh/2.0-rc1/api/paddle/nn/layer/rnn/LSTM_cn.html) API实现。如果读者对使用基础算子拼装LSTM的内容感兴趣，可以查阅[paddle.nn.LSTM](https://www.paddlepaddle.org.cn/documentation/docs/zh/2.0-rc1/api/paddle/nn/layer/rnn/LSTM_cn.html) 类的源代码。 

#### seq2vec的文本表示

在讲解情感分析网络搭建之前，我们先介绍一下seq2vec

句子情感分析的关键技术是如何将文本表示成一个**携带语义的文本向量**。随着深度学习技术的快速发展，目前常用的文本表示技术有LSTM，GRU，RNN等方法。
PaddleNLP提供了一系列的文本表示技术，集成在`seq2vec`模块中。

[`paddlenlp.seq2vec`](https://github.com/PaddlePaddle/models/tree/develop/PaddleNLP/paddlenlp/seq2vec) 模块的作用是将输入的序列文本，表示成一个语义向量。

<center><img src="../assets/03-lstm-sentiment-7.png" width="700" height="350" ></center>
<br><center>图10：paddlenlp.seq2vec示意图</center></br>


 本模块主要使用了seq2vec的LSTMEncoder部分，其核心结构是使用的LSTM，LSTMEncoder的核心是实现下面的内容：

+ LSTMEncoder：
    + get_input_dim : encoder 的输入的维度
    + get_output_dim : encoder 的输出维度
    + forward : 前向传播的逻辑
    
    
LSTMEncoder核心实现代码如下，这段代码只做帮助理解

In [91]:
class LSTMEncoder(nn.Layer):
    def __init__(self,
                 input_size,
                 hidden_size,
                 num_layers=1,
                 direction="forward",
                 dropout=0.0,
                 pooling_type=None,
                 **kwargs):
        super().__init__()
        self._input_size = input_size
        self._hidden_size = hidden_size
        self._direction = direction
        self._pooling_type = pooling_type
        # LSTM层
        self.lstm_layer = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            direction=direction,
            dropout=dropout,
            **kwargs)

    def get_input_dim(self):
        # 获取LSTM的输入的维度
        return self._input_size

    def get_output_dim(self):
        # 获取LSTM的输出的维度
        if self._direction == "bidirect":
            return self._hidden_size * 2
        else:
            return self._hidden_size

    def forward(self, inputs, sequence_length):
        encoded_text, (last_hidden, last_cell) = self.lstm_layer(
            inputs, sequence_length=sequence_length)
        # 如果是双向的LSTM，则输出最后一个时刻
        if self._direction != 'bidirect':
            # 输出最后一层的向量
            output = last_hidden[-1, :, :]
        else:
            # 单向的LSTM的实现，把倒数第一层和倒数第二层的向量拼接后输出
            output = paddle.concat(
                    (last_hidden[-2, :, :], last_hidden[-1, :, :]), axis=1)
        return output


#### 情感分析网络


<center><img src="../assets/03-lstm-sentiment-8.png" width="700px"></center>
<br><center>图11：情感分析网络</center>


上图显示的是情感分析网络，分为文本向量化，序列学习，特征学习，分类四部分：

1.文本向量化：这部分的作用就是把文本id的形式转换成稠密的向量的形式；

2.序列学习：序列学习是seq2vec层，具体使用的是双向LSTM用来学习文本序列的相关关系；

3.特征学习：特征学习使用了全连接层，主要是对前面的序列特征进一步学习，得到全局的特征信息；

4.分类：分类是输出层，本质上是一个全连接层，主要是把特征转化成情感类别概率输出。

情感分析网络代码实现如下：




In [92]:
import paddlenlp as ppnlp
import paddle.nn as nn
import paddle.nn.functional as F

class LSTMModel(nn.Layer):
    def __init__(self,
                 vocab_size,
                 num_classes,
                 emb_dim=300,  # 修改为300以匹配预训练模型维度
                 padding_idx=0,
                 lstm_hidden_size=198,
                 direction='forward',
                 lstm_layers=1,
                 dropout_rate=0.0,
                 pooling_type=None,
                 fc_hidden_size=96,
                 trainable_emb=True):  # 词向量不更新
        super().__init__()
        
        # 使用PaddleNLP预训练词向量替代随机初始化的Embedding
        # TokenEmbedding直接加载了在亿级文本上预训练好的词向量。这些词向量已经编码了丰富的语言知识
        self.embedder = ppnlp.embeddings.TokenEmbedding(
            embedding_name="w2v.baidu_encyclopedia.target.word-word.dim300",
            trainable=trainable_emb  # 控制词向量是否参与训练
        )
        
        # 获取预训练词向量的实际维度
        actual_emb_dim = self.embedder.embedding_dim

            
        
        # 序列学习 - 调整LSTM输入维度以匹配预训练词向量
        self.lstm_encoder = ppnlp.seq2vec.LSTMEncoder(
            actual_emb_dim,  # 使用实际维度
            lstm_hidden_size,
            num_layers=lstm_layers,
            direction=direction,
            dropout=dropout_rate,
            pooling_type=pooling_type)
        
        # 特征学习
        self.fc = nn.Linear(self.lstm_encoder.get_output_dim(), fc_hidden_size)
        
        # 输出层
        self.output_layer = nn.Linear(fc_hidden_size, num_classes)

    def forward(self, text, seq_len):
        # 输入的文本通过预训练词向量层
        # text应该是token IDs, shape为[batch_size, max_seq_len]
        embedded_text = self.embedder(text)  # 输出维度: [batch_size, max_seq_len, 300]
        
        # LSTM编码
        text_repr = self.lstm_encoder(embedded_text, sequence_length=seq_len)
        
        # 全连接层
        fc_out = paddle.tanh(self.fc(text_repr))
        
        # 输出层
        logits = self.output_layer(fc_out)
        probs = F.softmax(logits, axis=-1)
        return probs

In [93]:
# import paddlenlp as ppnlp

# class LSTMModel(nn.Layer):
#     def __init__(self,
#                  vocab_size,
#                  num_classes,
#                  emb_dim=128,
#                  padding_idx=0,
#                  lstm_hidden_size=198,
#                  direction='forward',
#                  lstm_layers=1,
#                  dropout_rate=0.0,
#                  pooling_type=None,
#                  fc_hidden_size=96):
#         super().__init__()
#         # 文本向量化
#         # 首先将输入word id 查表后映射成 word embedding
#         self.embedder = nn.Embedding(
#             num_embeddings=vocab_size,
#             embedding_dim=emb_dim,
#             padding_idx=padding_idx)
#         # 序列学习
#         # 将word embedding经过LSTMEncoder变换到文本语义表征空间中
#         self.lstm_encoder = ppnlp.seq2vec.LSTMEncoder(
#             emb_dim,
#             lstm_hidden_size,
#             num_layers=lstm_layers,
#             direction=direction,
#             dropout=dropout_rate,
#             pooling_type=pooling_type)
#         # 特征学习
#         # LSTMEncoder.get_output_dim()方法可以获取经过encoder之后的文本表示hidden_size
#         self.fc = nn.Linear(self.lstm_encoder.get_output_dim(), fc_hidden_size)
#         # 输出层
#         # 最后的分类器
#         self.output_layer = nn.Linear(fc_hidden_size, num_classes)

#      # forwad函数即为模型前向计算的函数，它有两个输入，分别为：
#     # input为输入的训练文本，其shape为[batch_size, max_seq_len]
#     # seq_len训练文本对应的真实长度，其shape维[batch_size]
#     def forward(self, text, seq_len):
#         # 输入的文本的维度(batch_size, num_tokens, embedding_dim)
#         embedded_text = self.embedder(text)
#         # lstm的输出的维度: (batch_size, num_tokens, num_directions*lstm_hidden_size)
#         # 如果lstm是双向的，则num_directions = 2，如果是单向的则num_directions的维度是1
#         text_repr = self.lstm_encoder(embedded_text, sequence_length=seq_len)
#         # 全连接层的的维度是(batch_size, fc_hidden_size)
#         fc_out = paddle.tanh(self.fc(text_repr))
#         # 输出层的维度(batch_size, num_classes)
#         logits = self.output_layer(fc_out)
#         # probs 分类概率值
#         probs = F.softmax(logits, axis=-1)
#         return probs

### 2.3.3 模型训练

在完成模型定义之后，我们就可以开始训练模型了。当训练结束以后，我们可以使用测试集合评估一下当前模型的效果，代码如下：

In [94]:
# 定义训练参数
epoch_num = 4
batch_size = 128

learning_rate = 5e-5
dropout_rate = 0.2
num_layers = 1
hidden_size = 256
embedding_size = 256
vocab_size=len(vocab)
print(vocab_size)
# 实例化LSTM模型
model= LSTMModel(
        vocab_size,
        num_classes=2,
        emb_dim=embedding_size,
        lstm_layers=num_layers,
        direction='bidirectional',
        padding_idx=vocab['[PAD]'])
# 指定优化策略，更新模型参数
optimizer = paddle.optimizer.Adam(learning_rate=learning_rate, beta1=0.9, beta2=0.999, parameters= model.parameters()) 

[2026-01-09 21:57:43,389] [    INFO] - Loading token embedding...


8911


[2026-01-09 21:57:45,960] [    INFO] - Finish loading embedding vector.
[2026-01-09 21:57:45,962] [    INFO] - Token Embedding info:             
Unknown index: 635963             
Unknown token: [UNK]             
Padding index: 635964             
Padding token: [PAD]             
Shape :[635965, 300]


In [95]:
paddle.seed(0)
random.seed(0)
np.random.seed(0)
# 定义训练函数
# 记录训练过程中的损失变化情况，可用于后续画图查看训练情况
losses = []
steps = []

def train(model):
    # 开启模型训练模式
    model.train()
    
    global_step =  0
    for epoch in range(epoch_num):
        for step, (sentences,valid_length, labels) in enumerate(train_loader):
        
            # 前向计算，将数据feed进模型，并得到预测的情感标签和损失
            logits = model(sentences,valid_length)

            # 计算损失
            loss = F.cross_entropy(input=logits, label=labels, soft_label=False)
            loss = paddle.mean(loss)

            # 后向传播
            loss.backward()
            # 更新参数
            optimizer.step()
            # 清除梯度
            optimizer.clear_grad()
            
            global_step+=1

            if global_step % 100 == 0:
                # 记录当前步骤的loss变化情况
                losses.append(loss.numpy())
                steps.append(step)
                # 打印当前loss数值
                print("epoch %d, step %d, loss %.3f" % (epoch,global_step, loss.numpy()))

#训练模型
train(model)
# 保存模型，包含两部分：模型参数和优化器参数
model_name = "sentiment_classifier"
# 保存训练好的模型参数
paddle.save(model.state_dict(), "checkpoint/{}.pdparams".format(model_name))
# 保存优化器参数，方便后续模型继续训练
paddle.save(optimizer.state_dict(), "checkpoint/{}.pdopt".format(model_name))

epoch 0, step 100, loss 0.681
epoch 1, step 200, loss 0.660
epoch 1, step 300, loss 0.615
epoch 2, step 400, loss 0.525
epoch 3, step 500, loss 0.520
epoch 3, step 600, loss 0.510


从上述的loss变化过程可以看到，loss是不断在降低的。

**tips**

使用GPU训练，设置epoch为20，训练2分钟左右，loss降低到0.313，大概1100个step之后就可以达到90.50%的准确率。训练的epoch越多，精度可能会更高，但不是绝对的会升高，会出现过拟合现象（训练集的loss降低，验证集的精度反而升高的现象）。




### 2.2.4 模型评估
在模型训练阶段，保存了训练完成的模型参数。因此在模型评估阶段，首先需要加载保存到磁盘的模型参数，在获得完整的模型之后，利用相应的测试集开始进行模型评估。


In [96]:
@paddle.no_grad()
def evaluate(model):
    # 开启模型测试模式，在该模式下，网络不会进行梯度更新
    model.eval()

    # 定义以上几个统计指标
    tp, tn, fp, fn = 0, 0, 0, 0
    
    for sentences,valid_lens, labels in test_loader:
    
        # 获取模型对当前batch的输出结果
        logits = model(sentences,valid_lens)
        
        # 使用softmax进行归一化
        probs = F.softmax(logits)

        # 把输出结果转换为numpy array数组，比较预测结果和对应label之间的关系，并更新tp，tn，fp和fn
        probs = probs.numpy()
        for i in range(len(probs)):
            # 当样本是的真实标签是正例
            if labels[i] == 1:
                # 模型预测是正例
                if probs[i][1] > probs[i][0]:
                    tp += 1
                # 模型预测是负例
                else:
                    fn += 1
            # 当样本的真实标签是负例
            else:
                # 模型预测是正例
                if probs[i][1] > probs[i][0]:
                    fp += 1
                # 模型预测是负例
                else:
                    tn += 1

    # 整体准确率
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    
    # 输出最终评估的模型效果
    print("TP: {}\nFP: {}\nTN: {}\nFN: {}\n".format(tp, fp, tn, fn))
    print("Accuracy: %.4f" % accuracy)

# 加载训练好的模型进行预测，重新实例化一个模型，然后将训练好的模型参数加载到新模型里面
state_dict=paddle.load('checkpoint/sentiment_classifier.pdparams')
model.load_dict(state_dict)
# 评估模型
evaluate(model)

TP: 470
FP: 142
TN: 465
FN: 123

Accuracy: 0.7792


从输出结果可以看到，模型的准确率达到了86%。TP数量是481，FP的数量是56，TN的数量是551，FN的数量是112.（每次运行结果稍有差异）



### 2.2.5 模型预测
模型预测部分就是把文本转换成Tensor形式的ID行时候，利用模型预测得到输出，然后选取概率最大值的索引就行了。

In [75]:
import random
import numpy as np
import paddle

# 随机选择一个样本
random_index = random.randint(0, len(test_ds) - 1)
text = test_ds[random_index]['text']

# label_map = {0: 'negative', 1: 'positive'}
label_map = {0: '负面评价', 1: '正面评价'}

# 文本转换成ID的形式
input_ids = tokenizer.encode(text)
valid_lens = len(input_ids)

# 转换成Tensor的形式
input_ids = paddle.to_tensor([input_ids])
valid_lens = paddle.to_tensor([valid_lens])

# 模型预测
probs = model(input_ids, valid_lens)

# 取概率最大值的ID
idx = np.argmax(probs, axis=-1)
idx = idx.tolist()

# 得到预测标签
labels = [label_map[i] for i in idx]

# 看看预测样例分类结果
print('随机样本索引: {}'.format(random_index))
print('Data: {} \nLabel: {}'.format(text, labels[0]))

随机样本索引: 788
Data: 我十一来住了两天，一进门就象给骗的感觉。进旅馆有几百米的泥路。是很普通的装饰。价格上没有一点优势。不是黄金周定房不能退我就马上走了。这里没有停车场，停车要到隔壁缴费停车。哪天有个广州车停在路边过夜给老板骂个半死。我们带了几个小孩去玩，有一点调皮。小孩给老板的妈妈骂的很刻薄。天啊！我花五百多一晚来住还要受这样的气。呵呵！真的象他的广告一样：有回家的感觉。“因为我回家总是给老妈叨唠” 
Label: 负面评价


In [ ]:
#!/usr/bin/env python
# coding: utf-8

import warnings
warnings.filterwarnings("ignore")

import os
import random
import numpy as np
import pandas as pd
import paddle
import paddle.nn as nn
import paddle.nn.functional as F
from paddlenlp.datasets import load_dataset
from paddlenlp.utils.downloader import get_path_from_url
from paddlenlp.data import Vocab, JiebaTokenizer, Stack, Pad, Tuple
from collections import defaultdict
import jieba
from functools import partial

# 设置GPU
paddle.set_device('gpu:0')

# ==================== 1. 数据加载与处理 ====================

# 下载数据集
URL = "https://bj.bcebos.com/paddlenlp/datasets/ChnSentiCorp.zip"
if not os.path.exists('ChnSentiCorp.zip'):
    get_path_from_url(URL, root_dir='../../data/')

# 数据读取函数
def read(split='train'):
    data_dict = {
        'train': '../../data/ChnSentiCorp/train.tsv',
        "dev": '../../data/ChnSentiCorp/dev.tsv',
        'test': '../../data/ChnSentiCorp/test.tsv'
    }
    with open(data_dict[split], 'r') as f:
        head = None
        for line in f.readlines():
            data = line.strip().split("\t")
            if not head:
                head = data
            else:
                if split == 'train':
                    label, text = data
                    yield {"text": text, "label": label, "qid": ''}
                elif split == 'dev':
                    qid, label, text = data
                    yield {"text": text, "label": label, "qid": qid}
                elif split == 'test':
                    qid, text = data
                    yield {"text": text, "label": '', "qid": qid}

# 加载数据集
train_ds = load_dataset(read, split="train", lazy=False)
dev_ds = load_dataset(read, split="dev", lazy=False)
test_ds = load_dataset(read, split="test", lazy=False)

# ==================== 2. 构建词表 ====================

def build_vocab(texts,
                stopwords=[],
                num_words=None,
                min_freq=10,
                unk_token="[UNK]",
                pad_token="[PAD]"):
    word_counts = defaultdict(int)
    for text in texts:
        if not text:
            continue
        # 使用结巴分词统计词频
        for word in jieba.cut(text):
            if word in stopwords:
                continue
            word_counts[word] += 1
    
    # 过滤低频词
    wcounts = []
    for word, count in word_counts.items():
        if count < min_freq:
            continue
        wcounts.append((word, count))
    
    # 按词频排序
    wcounts.sort(key=lambda x: x[1], reverse=True)
    
    # 限制词表大小
    if num_words is not None and len(wcounts) > (num_words - 2):
        wcounts = wcounts[:(num_words - 2)]
    
    # 构建词表
    sorted_voc = [pad_token, unk_token]
    sorted_voc.extend(wc[0] for wc in wcounts)
    word_index = dict(zip(sorted_voc, list(range(len(sorted_voc)))))
    return word_index

# 构建词汇表
texts = []
for data in train_ds:
    texts.append(data["text"])
for data in dev_ds:
    texts.append(data["text"])

# 停用词示例
stopwords = set(["的", "吗", "吧", "呀", "呜", "呢", "呗"])
word2idx = build_vocab(
    texts, stopwords, min_freq=5, unk_token="[UNK]", pad_token="[PAD]")
vocab = Vocab.from_dict(word2idx, unk_token="[UNK]", pad_token="[PAD]")

# ==================== 3. 数据预处理 ====================

def convert_example(example, tokenizer, is_test=False):
    # 将文本转换为ID序列
    input_ids = tokenizer.encode(example["text"])
    valid_length = np.array(len(input_ids), dtype='int64')
    input_ids = np.array(input_ids, dtype='int64')
    
    if not is_test:
        label = np.array(example["label"], dtype="int64")
        return input_ids, valid_length, label
    else:
        return input_ids, valid_length

# 初始化分词器
tokenizer = JiebaTokenizer(vocab)
trans_fn = partial(convert_example, tokenizer=tokenizer, is_test=False)

# 转换数据
train_ds = train_ds.map(trans_fn)
dev_ds = dev_ds.map(trans_fn)

# 数据批处理
batch_size = 64
batchify_fn = lambda samples, fn=Tuple(
    Pad(axis=0, pad_val=vocab.token_to_idx.get('[PAD]', 0)),
    Stack(dtype="int64"),
    Stack(dtype="int64")
): [data for data in fn(samples)]

train_sampler = paddle.io.BatchSampler(
    dataset=train_ds, batch_size=batch_size, shuffle=True)
test_sampler = paddle.io.BatchSampler(
    dataset=dev_ds, batch_size=batch_size, shuffle=True)

train_loader = paddle.io.DataLoader(
    train_ds, batch_sampler=train_sampler, collate_fn=batchify_fn)
test_loader = paddle.io.DataLoader(
    dev_ds, batch_sampler=test_sampler, collate_fn=batchify_fn)

# ==================== 4. 定义Word2Vec + LSTM模型 ====================

import paddlenlp as ppnlp

class Word2VecLSTMModel(nn.Layer):
    def __init__(self,
                 vocab_size,
                 num_classes,
                 emb_dim=300,  # 匹配预训练词向量维度
                 padding_idx=0,
                 lstm_hidden_size=198,
                 direction='forward',
                 lstm_layers=1,
                 dropout_rate=0.0,
                 pooling_type=None,
                 fc_hidden_size=96,
                 trainable_emb=False):  # 控制词向量是否可训练
        super().__init__()
        
        # 使用预训练的Word2Vec词向量
        self.embedder = ppnlp.embeddings.TokenEmbedding(
            embedding_name="w2v.baidu_encyclopedia.target.word-word.dim300",
            trainable=trainable_emb
        )
        
        # 获取实际词向量维度
        actual_emb_dim = self.embedder.embedding_dim
        
        # LSTM编码器
        self.lstm_encoder = ppnlp.seq2vec.LSTMEncoder(
            actual_emb_dim,
            lstm_hidden_size,
            num_layers=lstm_layers,
            direction=direction,
            dropout=dropout_rate,
            pooling_type=pooling_type)
        
        # 特征学习层
        self.fc = nn.Linear(self.lstm_encoder.get_output_dim(), fc_hidden_size)
        
        # 输出层
        self.output_layer = nn.Linear(fc_hidden_size, num_classes)

    def forward(self, text, seq_len):
        # 词嵌入
        embedded_text = self.embedder(text)  # [batch_size, max_seq_len, 300]
        
        # LSTM编码
        text_repr = self.lstm_encoder(embedded_text, sequence_length=seq_len)
        
        # 全连接层
        fc_out = paddle.tanh(self.fc(text_repr))
        
        # 输出层
        logits = self.output_layer(fc_out)
        probs = F.softmax(logits, axis=-1)
        return probs

# ==================== 5. 模型训练 ====================

# 定义训练参数
epoch_num = 4
batch_size = 128
learning_rate = 5e-5
dropout_rate = 0.2
num_layers = 1
hidden_size = 256

# 实例化模型
model = Word2VecLSTMModel(
    vocab_size=len(vocab),
    num_classes=2,
    lstm_hidden_size=hidden_size,
    lstm_layers=num_layers,
    direction='bidirectional',
    padding_idx=vocab['[PAD]'],
    trainable_emb=False  # 固定词向量，不参与训练
)

# 定义优化器
optimizer = paddle.optimizer.Adam(
    learning_rate=learning_rate,
    beta1=0.9,
    beta2=0.999,
    parameters=model.parameters()
)

# 设置随机种子
paddle.seed(0)
random.seed(0)
np.random.seed(0)

# 训练函数
def train(model):
    model.train()
    losses = []
    steps = []
    global_step = 0
    
    for epoch in range(epoch_num):
        for step, (sentences, valid_length, labels) in enumerate(train_loader):
            # 前向计算
            logits = model(sentences, valid_length)
            
            # 计算损失
            loss = F.cross_entropy(input=logits, label=labels, soft_label=False)
            loss = paddle.mean(loss)
            
            # 反向传播
            loss.backward()
            optimizer.step()
            optimizer.clear_grad()
            
            global_step += 1
            
            if global_step % 100 == 0:
                losses.append(loss.numpy())
                steps.append(step)
                print("epoch %d, step %d, loss %.3f" % (epoch, global_step, loss.numpy()))
    
    return losses, steps

# 开始训练
print("开始训练Word2Vec+LSTM模型...")
losses, steps = train(model)
print("训练完成!")

# 保存模型
model_name = "word2vec_lstm_classifier"
paddle.save(model.state_dict(), f"checkpoint/{model_name}.pdparams")
paddle.save(optimizer.state_dict(), f"checkpoint/{model_name}.pdopt")
print(f"模型已保存到 checkpoint/{model_name}.pdparams")

# ==================== 6. 模型评估 ====================

@paddle.no_grad()
def evaluate(model):
    model.eval()
    tp, tn, fp, fn = 0, 0, 0, 0
    
    for sentences, valid_lens, labels in test_loader:
        logits = model(sentences, valid_lens)
        probs = F.softmax(logits)
        probs = probs.numpy()
        
        for i in range(len(probs)):
            if labels[i] == 1:  # 真实标签为正例
                if probs[i][1] > probs[i][0]:  # 预测为正例
                    tp += 1
                else:  # 预测为负例
                    fn += 1
            else:  # 真实标签为负例
                if probs[i][1] > probs[i][0]:  # 预测为正例
                    fp += 1
                else:  # 预测为负例
                    tn += 1
    
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    print("TP: {}\nFP: {}\nTN: {}\nFN: {}\n".format(tp, fp, tn, fn))
    print("Word2Vec+LSTM模型准确率: {:.4f}".format(accuracy))
    return accuracy

print("\n评估模型性能...")
accuracy = evaluate(model)

# ==================== 7. 模型预测 ====================

def predict_single(text, model, tokenizer, label_map={0: '负面评价', 1: '正面评价'}):
    """单个文本预测"""
    model.eval()
    
    # 文本转换成ID序列
    input_ids = tokenizer.encode(text)
    valid_lens = len(input_ids)
    
    # 转换为Tensor
    input_ids = paddle.to_tensor([input_ids])
    valid_lens = paddle.to_tensor([valid_lens])
    
    # 模型预测
    with paddle.no_grad():
        probs = model(input_ids, valid_lens)
    
    # 获取预测结果
    idx = np.argmax(probs.numpy(), axis=-1)[0]
    predicted_label = label_map[idx]
    confidence = probs.numpy()[0][idx]
    
    return predicted_label, confidence, idx

def predict_batch(texts, model, tokenizer, label_map={0: '负面评价', 1: '正面评价'}):
    """批量文本预测"""
    model.eval()
    results = []
    
    for text in texts:
        input_ids = tokenizer.encode(text)
        valid_lens = len(input_ids)
        input_ids = paddle.to_tensor([input_ids])
        valid_lens = paddle.to_tensor([valid_lens])
        
        with paddle.no_grad():
            probs = model(input_ids, valid_lens)
        
        idx = np.argmax(probs.numpy(), axis=-1)[0]
        predicted_label = label_map[idx]
        confidence = probs.numpy()[0][idx]
        
        results.append({
            "text": text,
            "predicted_label": predicted_label,
            "confidence": confidence,
            "label_id": idx
        })
    
    return results

# ==================== 8. 测试预测功能 ====================

print("\n" + "="*50)
print("测试Word2Vec+LSTM模型预测功能")
print("="*50)

# 加载训练好的模型
print("加载训练好的模型...")
model_path = "checkpoint/word2vec_lstm_classifier.pdparams"
if os.path.exists(model_path):
    state_dict = paddle.load(model_path)
    model.load_dict(state_dict)
    print("模型加载成功!")
else:
    print("未找到训练好的模型，使用当前模型进行预测")

# 测试样本
test_samples = [
    "这个酒店的服务态度非常好，环境也很干净整洁，下次还会再来！",
    "房间太小了，而且卫生状况很差，床单都不干净，非常失望。",
    "位置很方便，交通便利，但是隔音效果太差了，晚上很吵。",
    "早餐种类丰富，味道也不错，性价比很高。",
    "前台服务太差了，等了半天都没人理，房间设施也很旧。"
]

print("\n预测结果：")
for i, text in enumerate(test_samples):
    label, confidence, label_id = predict_single(text, model, tokenizer)
    print(f"样本 {i+1}:")
    print(f"  文本: {text[:50]}...")
    print(f"  预测: {label} (置信度: {confidence:.4f})")
    print()

# ==================== 9. 与ERNIE模型对比 ====================

print("\n" + "="*50)
print("对比ERNIE模型和Word2Vec+LSTM模型")
print("="*50)

from paddlenlp.transformers import ErnieTokenizer, ErnieForSequenceClassification

# 加载ERNIE模型
print("加载ERNIE 3.0-Medium-zh模型...")
ernie_model_name = "ernie-3.0-medium-zh"
ernie_tokenizer = ErnieTokenizer.from_pretrained(ernie_model_name)
ernie_model = ErnieForSequenceClassification.from_pretrained(ernie_model_name, num_classes=2)
ernie_model.eval()

# 相同的测试文本对比
print("\nERNIE模型预测结果：")
for i, text in enumerate(test_samples[:3]):  # 只测试前3个样本
    # ERNIE编码
    encoded_input = ernie_tokenizer.encode(text)
    input_ids = paddle.to_tensor([encoded_input['input_ids']])
    token_type_ids = paddle.to_tensor([encoded_input['token_type_ids']])
    
    # ERNIE预测
    with paddle.no_grad():
        logits = ernie_model(input_ids, token_type_ids)
        probs = F.softmax(logits, axis=1)
    
    predicted_label_id = np.argmax(probs.numpy(), axis=-1)[0]
    predicted_label = "正面评价" if predicted_label_id == 1 else "负面评价"
    confidence = probs.numpy()[0][predicted_label_id]
    
    print(f"样本 {i+1}:")
    print(f"  文本: {text[:50]}...")
    print(f"  ERNIE预测: {predicted_label} (置信度: {confidence:.4f})")
    
    # Word2Vec+LSTM预测
    lstm_label, lstm_confidence, _ = predict_single(text, model, tokenizer)
    print(f"  LSTM预测: {lstm_label} (置信度: {lstm_confidence:.4f})")
    print()

# ==================== 10. 总结 ====================

print("\n" + "="*50)
print("模型总结")
print("="*50)
print(f"1. 模型类型: Word2Vec + BiLSTM")
print(f"2. 词向量: w2v.baidu_encyclopedia.target.word-word.dim300")

# 检查词向量是否可训练 - 修改后的方式
try:
    # 查看词向量层的参数是否设置了requires_grad
    embed_params = list(model.embedder.parameters())
    if embed_params:
        is_trainable = embed_params[0].stop_gradient == False
        print(f"3. 词向量是否可训练: {is_trainable}")
    else:
        print(f"3. 词向量是否可训练: 无法确定（预训练词向量通常固定）")
except Exception as e:
    print(f"3. 词向量是否可训练: 检查失败 - {str(e)}")

print(f"4. LSTM类型: 双向LSTM")
print(f"5. LSTM隐藏层维度: {hidden_size}")
print(f"6. 词汇表大小: {len(vocab)}")
print(f"7. 训练epoch数: {epoch_num}")
print(f"8. 最终准确率: {accuracy:.4f}")
print(f"9. 模型保存路径: checkpoint/{model_name}.pdparams")

print("\n注意：")
print("1. Word2Vec+LSTM模型在中小规模数据上表现良好")
print("2. 预训练词向量提供了良好的语义初始化")
print("3. LSTM能够捕获文本序列信息")
print("4. 相比ERNIE，参数量更少，训练更快")
print("5. 适合资源受限或对推理速度有要求的场景")

# # 或者在模型定义中明确记录trainable状态
# print("\n" + "="*50)
# print("模型详细信息")
# print("="*50)

# # 打印模型参数统计
# total_params = sum(p.numel() for p in model.parameters())
# trainable_params = sum(p.numel() for p in model.parameters() if not p.stop_gradient)
# print(f"总参数量: {total_params:,}")
# print(f"可训练参数量: {trainable_params:,}")
# print(f"冻结参数量: {total_params - trainable_params:,}")

# # 检查不同层的可训练状态
# print("\n各层可训练状态：")
# for name, param in model.named_parameters():
#     is_trainable = not param.stop_gradient
#     print(f"  {name}: {'可训练' if is_trainable else '冻结'} (形状: {param.shape})")

# 分析预测结果对比
print("\n" + "="*50)
print("模型预测对比分析")
print("="*50)

print("ERNIE和LSTM预测不一致的样本分析：")
print("样本1: '这个酒店的服务态度非常好，环境也很干净整洁，下次还会再来！'")
print("  - ERNIE预测: 负面评价 (置信度: 0.5854)")
print("  - LSTM预测: 正面评价 (置信度: 0.6966)")
print("  分析: ERNIE的置信度较低(0.5854)，接近随机猜测(0.5)，说明ERNIE对这个样本不太确定")
print("        LSTM预测为正面，置信度0.6966，较为确定")

print("\n样本3: '位置很方便，交通便利，但是隔音效果太差了，晚上很吵。'")
print("  - ERNIE预测: 负面评价 (置信度: 0.5928)")
print("  - LSTM预测: 正面评价 (置信度: 0.5497)")
print("  分析: 这是一个混合情感样本，前半句正面，后半句负面")
print("        ERNIE倾向于负面(可能是'太差了'起主导作用)")
print("        LSTM也接近随机猜测(0.5497)，说明模型对这个样本都不太确定")

print("\n准确率分析：")
print(f"Word2Vec+LSTM准确率: {accuracy:.2%}")
print("模型表现分析：")
print("1. 准确率69.83%，在情感分析任务中属于中等水平")
print("2. 对于明显的情感表达（样本2、4、5）预测置信度较高(>0.85)")
print("3. 对于混合情感或复杂表达（样本3）预测能力有限")
print("4. 相比预训练的ERNIE模型，LSTM在简单情感表达上表现尚可")

# 模型优化建议
print("\n" + "="*50)
print("模型优化建议")
print("="*50)
print("1. 增加训练数据量")
print("2. 尝试使用双向LSTM（BiLSTM）")
print("3. 调整超参数（学习率、隐藏层维度等）")
print("4. 增加LSTM层数")
print("5. 使用注意力机制增强模型")
print("6. 尝试不同的优化器")
print("7. 调整词向量的可训练性（trainable_emb=True）")
print("8. 增加Dropout防止过拟合")
print("9. 使用更复杂的模型结构（如CNN+LSTM）")
print("10. 尝试集成学习方法")

print("\nWord2Vec+LSTM情感分析模型完整流程演示完毕！")